# 01 — EDA: the Edmunds car-reviews dataset and the label proxy

**Descriptive pillar.** Before any modeling, two questions: what does the
review data look like, and is the rating-derived sentiment label honest enough
to evaluate the Triage skill against?

Source: [`florentgbelidji/edmunds-car-ratings`](https://huggingface.co/datasets/florentgbelidji/edmunds-car-ratings)
— public, no auth. The honesty boundary (see `docs/requirement.md`): the
sentiment label is a **proxy** from the star rating, not a human annotation.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from src.data import load_labeled_reviews, label_balance, train_eval_split, rating_to_label

df = load_labeled_reviews()
print(f'{len(df):,} labeled reviews (3-star middle dropped as ambiguous)')
df.head()

## The label proxy, stated plainly

`Rating >= 4 -> POSITIVE`, `Rating <= 2 -> NEGATIVE`, `Rating == 3 -> dropped`.
A 5-star review that says "great car, terrible dealership" is noise the proxy
cannot resolve. We measure agreement with *rating-implied* sentiment, and label
it as such everywhere.

In [ ]:
balance = label_balance(df)
print('Label balance:', balance)
# The class skew is WHY the acceptance metric is macro-F1, not accuracy:
# a model that always predicts the majority class would score high accuracy
# while being useless for triage (catching unhappy customers).

In [ ]:
df['n_chars'] = df['review'].str.len()
print(df['n_chars'].describe())
# Review length distribution motivates the Digest skill: long reviews are
# exactly what agents don't have time to read.

In [ ]:
train_df, eval_df = train_eval_split(df)
print(f'dev slice: {len(train_df):,}   held-out eval: {len(eval_df):,}')
print('eval balance:', label_balance(eval_df))
# The split is stratified and seeded so the eval set is identical across runs.